In [84]:
import numpy as np
import pandas as pd

import os
import pickle

from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from statistics import median

from datetime import datetime

from sklearn.model_selection import train_test_split

# Load data

In [87]:
df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/cohort_covariates_final.csv')

# Fill null AKI values with 0 and create binary AKI columns

We assume that if a value indicating if a patient has AKI is missing, then that patient does not have AKI.

In [8]:
df['aki_12hrs'].fillna(0, inplace=True)
df['aki_72hrs'].fillna(0, inplace=True)

df['aki_12hrs_any'] = [1 if aki > 0 else 0 for aki in df['aki_12hrs']]
df['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in df['aki_72hrs']]

In [17]:
df['Medical.LOS.days'] = df['Medical.LOS'] / 24

# Remove patients with AKI at 12 hours

In [66]:
print('Number of total patient-stays in data: %d' % len(df))
print('Number of patient-stays where patient develops AKI by hour 12: %d' % len(df[df['aki_12hrs'] > 0]))

Number of total patient-stays in data: 59330
Number of patient-stays where patient develops AKI by hour 12: 3440


In [70]:
df = df[df['aki_12hrs'] == 0]

# Remove patients with length of stay less than 12 hours

In [ ]:
df = df[df['Physical.LOS'] >= 0.5]

# len(df[df['Physical.LOS'] >= 0.5])

55890

# Check AKI stage counts

In [28]:
df['aki_72hrs'].value_counts()

aki_72hrs
0.0    54650
1.0      762
2.0      265
3.0      213
Name: count, dtype: int64

In [51]:
265 + 213

478

In [53]:
478 / len(df)

0.008552513866523528

In [55]:
762 / len(df)

0.013633923778851314

# Get unique patient count

In [82]:
df['person_id'].nunique()

36482

# Get statistics for all columns now that patients with AKI at 12 hours are filtered out

In [107]:
df['Outcome'].value_counts()

Outcome
Survived    55182
Died          708
Name: count, dtype: int64

In [60]:
vs_features = [
    'dbp_min',
    'hr_min',
    'mbp_min',
    'rr_min',
    'sbp_min',
    'spo2_min',
    'temp_min',
    'fio2_min'
]

for feature in vs_features:
    print(feature.replace('_min', ''))
    print('Occurrence count = %d' % (len(df) - df[feature].isna().sum()))
    print('Occurrence proportion = %.3f' % ((len(df) - df[feature].isna().sum()) / len(df)))

dbp
Occurrence count = 53430
Occurrence proportion = 0.956
hr
Occurrence count = 48394
Occurrence proportion = 0.866
mbp
Occurrence count = 51840
Occurrence proportion = 0.928
rr
Occurrence count = 49544
Occurrence proportion = 0.886
sbp
Occurrence count = 53430
Occurrence proportion = 0.956
spo2
Occurrence count = 54385
Occurrence proportion = 0.973
temp
Occurrence count = 46145
Occurrence proportion = 0.826
fio2
Occurrence count = 3966
Occurrence proportion = 0.071


# Remove lab features with less than 500 non-null values

In [63]:
na_count_dict_list = []

for col in df.columns:
    col_na_proportion = len(df[df[col].isna()]) / len(df)
    temp_dict = {
        'feature': col,
        'na_prop': col_na_proportion,
        'not_null_count': len(df) - len(df[df[col].isna()])
    }
    na_count_dict_list.append(temp_dict)

na_count_df = pd.DataFrame(na_count_dict_list)

In [64]:
na_count_df.sort_values('na_prop', ascending=False).head(15)

,feature,na_prop,not_null_count
49,bcx_bact_min,1.000000,0
87,bcx_bact_max,1.000000,0
53,ucx_bact_min,1.000000,0
129,ucx_bact_mean,1.000000,0
163,bcx_bact_median,1.000000,0
125,bcx_bact_mean,1.000000,0
206,bcx_bact_latest,1.000000,0
167,ucx_bact_median,1.000000,0
242,ucx_bact_latest,1.000000,0
91,ucx_bact_max,1.000000,0


In [65]:
cols_to_drop = list(na_count_df[na_count_df['not_null_count'] < 500]['feature'])
cols_to_drop = [col for col in cols_to_drop if 'latest' not in col]
        
cols_to_drop

['viral_min',
 'bcx_bact_min',
 'ucx_bact_min',
 'viral_max',
 'bcx_bact_max',
 'ucx_bact_max',
 'viral_mean',
 'bcx_bact_mean',
 'ucx_bact_mean',
 'viral_median',
 'bcx_bact_median',
 'ucx_bact_median']

In [66]:
df.drop(cols_to_drop, axis=1, inplace=True)

# Perform one-hot encoding

## Gender

In [68]:
gender_1h_df = pd.get_dummies(df['Gender'], dtype=int)
# gender_1h_df['Gender_Ambiguous'] = gender_1h_df['Ambiguous']
gender_1h_df.drop('Ambiguous', axis=1, inplace=True)

df = pd.merge(df, gender_1h_df, left_index=True, right_index=True)
df.drop('Gender', axis=1, inplace=True)

In [99]:
df['Female'].sum()

24679

## Nephrotoxic Medication Count

In [70]:
df['nephrotoxic_med_type_count_eq_0'] = [1 if count == 0 else 0 for count in df['nephrotoxic_med_type_count']]
df['nephrotoxic_med_type_count_eq_1'] = [1 if count == 1 else 0 for count in df['nephrotoxic_med_type_count']]
df['nephrotoxic_med_type_count_eq_2'] = [1 if count == 2 else 0 for count in df['nephrotoxic_med_type_count']]
df['nephrotoxic_med_type_count_gt_eq_3'] = [1 if count >= 3 else 0 for count in df['nephrotoxic_med_type_count']]

df['nephrotoxic_med_type_count_gt_0'] = [1 if count > 0 else 0 for count in df['nephrotoxic_med_type_count']]

# Save dataset of latest lab measurements by patient and lab type

These will be used for imputation along with median lab result values if latest lab result is also null.

In [ ]:
id_cols = ['CaseIndex', 'adt_datetime']
latest_lab_result_cols = [col for col in df.columns if 'latest' in col]

latest_lab_result_df = df[id_cols + latest_lab_result_cols].copy()

latest_lab_result_df.to_csv(os.getenv('AKI_CSV_DIR') + '/latest_lab_results_data.csv', index=False)

# Split data into analysis and holdout sets

In [74]:
def get_datetime_obj_from_str(dt_str):
    year = int(dt_str.split('-')[0])
    month = int(dt_str.split('-')[1])
    day = int(dt_str.split('-')[2].split(' ')[0])
    try:
        hour = int(dt_str.split(' ')[1].split(':')[0])
        minute = int(dt_str.split(' ')[1].split(':')[1])
        second = int(dt_str.split(':')[-1])
    except:
        hour = 0
        minute = 0
        second = 0

    dt_obj = datetime(year, month, day, hour, minute, second)

    return dt_obj

In [75]:
df['adt_datetime_obj'] = [get_datetime_obj_from_str(dt_str) for dt_str in df['adt_datetime']]

cutoff_datetime = datetime(2016, 6, 1, 0, 0, 0)

holdout_df = df[df['adt_datetime_obj'] >= cutoff_datetime].copy()
analysis_df = df[df['adt_datetime_obj'] < cutoff_datetime].copy()

holdout_df.drop('adt_datetime_obj', axis=1, inplace=True)
analysis_df.drop('adt_datetime_obj', axis=1, inplace=True)

In [76]:
print('Number of AKI encounters in holdout dataset: %d' % len(holdout_df[holdout_df['aki_72hrs'] > 0]))
print('Total number of encounters in holdout dataset: %d' % len(holdout_df))
print('Number of AKI encounters in analysis dataset: %d' % len(analysis_df[analysis_df['aki_72hrs'] > 0]))
print('Total number of encounters in analysis dataset: %d' % len(analysis_df))

Number of AKI encounters in holdout dataset: 303
Total number of encounters in holdout dataset: 14264
Number of AKI encounters in analysis dataset: 937
Total number of encounters in analysis dataset: 41626


# Save identifier columns

In [ ]:
analysis_df[['CaseIndex', 
             'pn.site', 
             'vps.site',
             'hashid',
             'adt_datetime']].to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_data_IDs.csv', index=False)
holdout_df[['CaseIndex', 
            'pn.site',
            'vps.site',
             'hashid',
             'adt_datetime']].to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_data_IDs.csv', index=False)

# Split analysis into train and validate sets and save identifying columns (CaseIndex and adt_datetime)

In [80]:
train_df, val_df = train_test_split(analysis_df, 
                                    test_size=0.2,
                                    stratify=analysis_df['aki_72hrs_any'],
                                    random_state=343)

In [81]:
train_IDs_df = train_df[['CaseIndex', 'adt_datetime']]
val_IDs_df = val_df[['CaseIndex', 'adt_datetime']]

In [101]:
print('Encounters in train set: %d (%.2f)' % (len(train_IDs_df), len(train_IDs_df) / (len(df))))
print('Encounters in validation set: %d (%.2f)' % (len(val_IDs_df), len(val_IDs_df) / (len(df))))

Encounters in train set: 33300 (0.60)
Encounters in validation set: 8326 (0.15)


In [ ]:
train_IDs_df.to_csv(os.getenv('AKI_CSV_DIR') + '/train_set_IDs.csv', index=False)
val_IDs_df.to_csv(os.getenv('AKI_CSV_DIR') + '/val_set_IDs.csv', index=False)

# Find median of each feature used in analysis

In [21]:
non_feature_cols = [
    'pn.site',
    'vps.site',
    'CaseIndex',
    'adt_datetime',
    'medical_dc_dt',
    'Outcome',
    'person_id',
    'Medical.LOS',
    'Physical.LOS',
    'hashid',
    'baseline_used'
]

modeling_feature_list = []

for col in analysis_df.columns:
    if col not in non_feature_cols and 'latest' not in col:
        modeling_feature_list.append(col)

save_feature_list = False
if save_feature_list:
    with open('pickle/full_modeling_feature_list_including_bSCr.pickle', 'wb') as outfile:
        pickle.dump(modeling_feature_list, outfile)

In [22]:
feature_medians_dict_list = []
feature_medians_dict = dict()

for feature in modeling_feature_list:
    try:
        feature_median = median(analysis_df[feature].dropna())
        temp_dict = {
            'feature': feature,
            'median': feature_median
        }
        feature_medians_dict_list.append(temp_dict)
        feature_medians_dict[feature] = feature_median
    except:
        print('Error on feature: ' + feature)

feature_medians_df = pd.DataFrame(feature_medians_dict_list)
feature_medians_df.head(10)

,feature,median
0,age,4.33
1,bSCr_prior,0.30
2,baseline_bSCr,0.38
3,aki_12hrs,0.00
4,aki_72hrs,0.00
5,anc_min,140.00
6,be_min,-3.00
7,bicarb_min,21.00
8,Blood Urea Nitrogen_min,7.00
9,cl_min,106.00


In [23]:
feature_medians_df.to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_feature_medians_including_baseline_bSCr.csv', index=False)

save_medians = False
if save_medians:
    with open('pickle/analysis_data_feature_median_dict_including_baseline_bSCr.pickle', 'wb') as outfile:
        pickle.dump(feature_medians_dict, outfile)

# Save un-imputed analysis and holdout sets

In [24]:
analysis_df['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in analysis_df['aki_72hrs_any']]
holdout_df['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in holdout_df['aki_72hrs_any']]

In [25]:
analysis_df.to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_includes_bSCr.csv', index=False)
holdout_df.to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_includes_bSCr.csv', index=False)

# Make copies of datasets with latest lab results imputed if no lab results in time window of interest

In [25]:
analysis_df_latest_lab_imputed = analysis_df.copy()
holdout_df_latest_lab_imputed = holdout_df.copy()

In [26]:
latest_lab_cols = [col for col in df.columns if 'latest' in col]
# print(len(latest_lab_cols))

for col in latest_lab_cols:
    lab_type = col.replace('_latest', '')
    for agg_type in ['min', 'max', 'mean', 'median']:
        if (lab_type + '_' + agg_type) in analysis_df_latest_lab_imputed.columns:
            analysis_df_latest_lab_imputed[lab_type + '_' + agg_type].fillna(analysis_df_latest_lab_imputed[col], inplace=True)
            holdout_df_latest_lab_imputed[lab_type + '_' + agg_type].fillna(holdout_df_latest_lab_imputed[col], inplace=True)
# for col in df.columns:
#     if 'latest' in col:
#         df.drop(col, axis=1, inplace=True)

In [27]:
analysis_df_latest_lab_imputed['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in analysis_df_latest_lab_imputed['aki_72hrs_any']]
holdout_df_latest_lab_imputed['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in holdout_df_latest_lab_imputed['aki_72hrs_any']]

analysis_df_latest_lab_imputed.to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_imputed_includes_bSCr.csv', index=False)
holdout_df_latest_lab_imputed.to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_latest_lab_imputed_includes_bSCr.csv', index=False)

# Impute values that are still missing with median

In [28]:
analysis_df_fully_imputed = analysis_df_latest_lab_imputed.copy()
holdout_df_fully_imputed = holdout_df_latest_lab_imputed.copy()

analysis_df_median_imputed_only = analysis_df.copy()
holdout_df_median_imputed_only = holdout_df.copy()

In [29]:
for col in analysis_df_fully_imputed.columns:
    if col in feature_medians_dict.keys():
        analysis_df_fully_imputed[col].fillna(feature_medians_dict[col], inplace=True)

for col in holdout_df_fully_imputed.columns:
    if col in feature_medians_dict.keys():
        holdout_df_fully_imputed[col].fillna(feature_medians_dict[col], inplace=True)

for col in analysis_df_median_imputed_only.columns:
    if col in feature_medians_dict.keys():
        analysis_df_median_imputed_only[col].fillna(feature_medians_dict[col], inplace=True)

for col in holdout_df_median_imputed_only.columns:
    if col in feature_medians_dict.keys():
        holdout_df_median_imputed_only[col].fillna(feature_medians_dict[col], inplace=True)

In [31]:
analysis_df_fully_imputed['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in analysis_df_fully_imputed['aki_72hrs_any']]
holdout_df_fully_imputed['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in holdout_df_fully_imputed['aki_72hrs_any']]

analysis_df_median_imputed_only['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in analysis_df_fully_imputed['aki_72hrs_any']]
holdout_df_median_imputed_only['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in holdout_df_fully_imputed['aki_72hrs_any']]

analysis_df_fully_imputed.to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed.csv', index=False)
holdout_df_fully_imputed.to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_latest_lab_and_median_imputed.csv', index=False)

analysis_df_median_imputed_only.to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_only_includes_bSCr.csv', index=False)
holdout_df_median_imputed_only.to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_median_imputed_only_includes_bSCr.csv', index=False)

# Perform min-max normalization

In [32]:
analysis_df_fully_imputed_normalized = analysis_df_fully_imputed.copy()
holdout_df_fully_imputed_normalized = holdout_df_fully_imputed.copy()

analysis_df_median_imputed_normalized = analysis_df_median_imputed_only.copy()
holdout_df_median_imputed_normalized = holdout_df_median_imputed_only.copy()

In [33]:
min_val_dict = dict()
max_val_dict = dict()

for feature in modeling_feature_list:
    analysis_feature_min = analysis_df_fully_imputed_normalized[feature].min()
    analysis_feature_max = analysis_df_fully_imputed_normalized[feature].max()

    min_val_dict[feature] = analysis_feature_min
    max_val_dict[feature] = analysis_feature_max
    
    analysis_df_fully_imputed_normalized[feature] = [(x - analysis_feature_min) / (analysis_feature_max - analysis_feature_min + 1e-8) for x in analysis_df_fully_imputed_normalized[feature]]
    analysis_df_median_imputed_normalized[feature] = [(x - analysis_feature_min) / (analysis_feature_max - analysis_feature_min + 1e-8) for x in analysis_df_median_imputed_normalized[feature]]
    
    holdout_feature_min = holdout_df_fully_imputed_normalized[feature].min()
    holdout_feature_max = holdout_df_fully_imputed_normalized[feature].max()
    holdout_df_fully_imputed_normalized[feature] = [(x - holdout_feature_min) / (holdout_feature_max - holdout_feature_min + 1e-8) for x in holdout_df_fully_imputed_normalized[feature]]
    holdout_df_median_imputed_normalized[feature] = [(x - holdout_feature_min) / (holdout_feature_max - holdout_feature_min + 1e-8) for x in holdout_df_median_imputed_normalized[feature]]

    
    # analysis_df_fully_imputed_normalized[feature] = [(x - analysis_feature_min) / (analysis_feature_max - analysis_feature_min + 1e-8) for x in analysis_df_fully_imputed_normalized[feature]]
    
    # holdout_feature_min = holdout_df_fully_imputed_normalized[feature].min()
    # holdout_feature_max = holdout_df_fully_imputed_normalized[feature].max()
    # holdout_df_fully_imputed_normalized[feature] = [(x - holdout_feature_min) / (holdout_feature_max - holdout_feature_min + 1e-8) for x in holdout_df_fully_imputed_normalized[feature]]

In [35]:
with open('min_val_dict.pickle', 'wb') as outfile:
    pickle.dump(min_val_dict, outfile)

with open('max_val_dict.pickle', 'wb') as outfile:
    pickle.dump(max_val_dict, outfile)

In [37]:
print('{')

for key in min_val_dict.keys():
    print('    "' + key + '": ' + str(min_val_dict[key]) + ',')

print('}')

{
    "age": 0.0,
    "bSCr_prior": 0.07,
    "baseline_bSCr": 0.0,
    "aki_12hrs": 0.0,
    "aki_72hrs": 0.0,
    "anc_min": 0.0,
    "be_min": -35.2,
    "bicarb_min": 1.0,
    "Blood Urea Nitrogen_min": 1.0,
    "cl_min": 58.0,
    "cr_min": 0.07,
    "crp_min": 0.07,
    "fibr_min": 41.0,
    "hct_min": 5.8,
    "hgb_min": 1.4,
    "k_min": 1.0,
    "lact_min": 0.3,
    "na_min": 6.0,
    "pco2_min": 6.0,
    "pH_min": 6.547,
    "Platelet Count_min": 0.0,
    "wbc_min": 0.0,
    "gluc_min": 7.0,
    "ical_min": 0.25,
    "po2_min": 6.0,
    "alb_min": 0.6,
    "alkphos_min": 9.0,
    "alt_min": 3.0,
    "ast_min": 5.0,
    "cal_min": 1.4,
    "cbili_min": 0.0,
    "ibili_min": 0.0,
    "mag_min": 0.2,
    "phos_min": 0.3,
    "prot_min": 2.0,
    "Total Bilirubin_min": 0.0,
    "inr_min": 0.63,
    "pt_min": 8.5,
    "segs_min": 0.0,
    "uprot_min": 5.8,
    "anc_max": 0.0,
    "be_max": -30.0,
    "bicarb_max": 5.0,
    "Blood Urea Nitrogen_max": 1.0,
    "cl_max": 74.0,
    "c

In [38]:
print('{')

for key in max_val_dict.keys():
    print('    "' + key + '": ' + str(max_val_dict[key]) + ',')

print('}')

{
    "age": 18.0,
    "bSCr_prior": 23.5,
    "baseline_bSCr": 1.86,
    "aki_12hrs": 0.0,
    "aki_72hrs": 3.0,
    "anc_min": 223290.0,
    "be_min": 30.0,
    "bicarb_min": 53.0,
    "Blood Urea Nitrogen_min": 107.0,
    "cl_min": 144.0,
    "cr_min": 13.2,
    "crp_min": 145.0,
    "fibr_min": 1381.0,
    "hct_min": 72.2,
    "hgb_min": 15927.0,
    "k_min": 9.0,
    "lact_min": 14.7,
    "na_min": 176.0,
    "pco2_min": 137.0,
    "pH_min": 7.65,
    "Platelet Count_min": 1070.0,
    "wbc_min": 70625.0,
    "gluc_min": 854.0,
    "ical_min": 9.2,
    "po2_min": 600.0,
    "alb_min": 5280.0,
    "alkphos_min": 9162.0,
    "alt_min": 16348.0,
    "ast_min": 21968.0,
    "cal_min": 15.6,
    "cbili_min": 35.9,
    "ibili_min": 22.3,
    "mag_min": 5.1,
    "phos_min": 21.75,
    "prot_min": 561.04,
    "Total Bilirubin_min": 34.6,
    "inr_min": 9.74,
    "pt_min": 76.3,
    "segs_min": 100.0,
    "uprot_min": 2400.0,
    "anc_max": 224751.0,
    "be_max": 33.7,
    "bicarb_max": 77

In [36]:
len(analysis_df_fully_imputed_normalized) + len(holdout_df_fully_imputed_normalized)

55890

In [35]:
latest_lab_result_cols = [col for col in analysis_df_fully_imputed_normalized.columns if 'latest' in col]

analysis_df_fully_imputed_normalized['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in analysis_df_fully_imputed_normalized['aki_72hrs_any']]
holdout_df_fully_imputed_normalized['aki_72hrs_any'] = [1 if aki > 0 else 0 for aki in holdout_df_fully_imputed_normalized['aki_72hrs_any']]

analysis_df_fully_imputed_normalized.drop(latest_lab_result_cols, axis=1).to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv', index=False)
holdout_df_fully_imputed_normalized.drop(latest_lab_result_cols, axis=1).to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv', index=False)

analysis_df_median_imputed_normalized.drop(latest_lab_result_cols, axis=1).to_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_normalized_includes_bSCr.csv', index=False)
holdout_df_median_imputed_normalized.drop(latest_lab_result_cols, axis=1).to_csv(os.getenv('AKI_CSV_DIR') + '/holdout_dataset_median_imputed_normalized_includes_bSCr.csv', index=False)